In [2]:
import duckdb

con = duckdb.connect("../olist.duckdb")

In [3]:
con.sql("SHOW TABLES").show()

┌───────────────────────────────────────┐
│                 name                  │
│                varchar                │
├───────────────────────────────────────┤
│ clean_customers                       │
│ clean_order_items                     │
│ clean_order_payments                  │
│ clean_order_reviews                   │
│ clean_orders                          │
│ clean_products                        │
│ raw_customers                         │
│ raw_geolocation                       │
│ raw_order_items                       │
│ raw_order_payments                    │
│ raw_order_reviews                     │
│ raw_orders                            │
│ raw_product_category_name_translation │
│ raw_products                          │
│ raw_sellers                           │
└───────────────────────────────────────┘
                 15 rows               



In [4]:
con.sql("""
    CREATE OR REPLACE TABLE clean_orders AS
    SELECT *
    FROM raw_orders
""")

CountBIGINT
99441


In [5]:
con.sql("""
    SELECT
        (SELECT COUNT(*) FROM raw_orders) AS raw_rows,
        (SELECT COUNT(*) FROM clean_orders) AS clean_rows
""").show()

┌──────────┬────────────┐
│ raw_rows │ clean_rows │
│  int64   │   int64    │
├──────────┼────────────┤
│    99441 │      99441 │
└──────────┴────────────┘



In [6]:
payments = con.sql("""
    SELECT *
    FROM raw_order_payments
""").df()

In [7]:
payments.shape

(103886, 5)

In [8]:
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [9]:
payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [10]:
payments[
    (payments['payment_value'] == 0) &
    (payments['payment_type'] == 'not_defined')
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [11]:
payments = payments[
    ~(
        (payments['payment_value'] == 0) &
        (payments['payment_type'] == 'not_defined')
    )
].copy()

In [12]:
payments.shape

(103883, 5)

In [13]:
con.register("payments_cleaned", payments)

con.sql("""
    CREATE OR REPLACE TABLE clean_order_payments AS
    SELECT *
    FROM payments_cleaned
""")

CountBIGINT
103883


In [14]:
con.sql("""
    SELECT COUNT(*) AS clean_rows
    FROM clean_order_payments
""").show()

┌────────────┐
│ clean_rows │
│   int64    │
├────────────┤
│     103883 │
└────────────┘



In [15]:
products = con.sql("""
    SELECT *
    FROM raw_products
""").df()

In [16]:
products['product_category_name'].isnull().sum()

np.int64(610)

In [17]:
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45,67,2,12300,40,40,40
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41,971,1,1700,16,19,16
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50,799,1,1400,27,7,27
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60,156,2,700,31,13,20


In [18]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

In [19]:
products['product_category_name'].isnull().sum()

np.int64(0)

In [20]:
products['product_category_name'='unknown'].value_counts()

SyntaxError: cannot assign to literal here. Maybe you meant '==' instead of '='? (483672492.py, line 1)

In [21]:
products['product_category_name'].value_counts().get('unknown', 0)

np.int64(610)

In [22]:
products[products['product_category_name'] == 'unknown'].shape[0]

610

In [23]:
(products['product_category_name'] == 'unknown').sum()

np.int64(610)

In [24]:
products[products['product_category_name'] == 'unknown']

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,unknown,<NA>,<NA>,<NA>,650,17,14,12
128,d8dee61c2034d6d075997acef1870e9b,unknown,<NA>,<NA>,<NA>,300,16,7,20
145,56139431d72cd51f19eb9f7dae4d1617,unknown,<NA>,<NA>,<NA>,200,20,20,20
154,46b48281eb6d663ced748f324108c733,unknown,<NA>,<NA>,<NA>,18500,41,30,41
197,5fb61f482620cb672f5e586bb132eae9,unknown,<NA>,<NA>,<NA>,300,35,7,12
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,unknown,<NA>,<NA>,<NA>,1800,30,20,70
32589,10dbe0fbaa2c505123c17fdc34a63c56,unknown,<NA>,<NA>,<NA>,800,30,10,23
32616,bd2ada37b58ae94cc838b9c0569fecd8,unknown,<NA>,<NA>,<NA>,200,21,8,16
32772,fa51e914046aab32764c41356b9d4ea4,unknown,<NA>,<NA>,<NA>,1300,45,16,45


In [25]:
con.register("products_cleaned", products)

con.sql("""
    CREATE OR REPLACE TABLE clean_products AS
    SELECT *
    FROM products_cleaned
""")

CountBIGINT
32951


In [26]:
con.sql("""
    SELECT COUNT(*) AS clean_rows
    FROM clean_products
""").show()

┌────────────┐
│ clean_rows │
│   int64    │
├────────────┤
│      32951 │
└────────────┘



In [27]:
con.sql("""
    CREATE OR REPLACE TABLE clean_order_reviews AS
    SELECT *
    FROM raw_order_reviews
""")

CountBIGINT
99224


In [28]:
con.sql("""
    SELECT COUNT(*) AS clean_rows
    FROM clean_order_reviews
""").show()

┌────────────┐
│ clean_rows │
│   int64    │
├────────────┤
│      99224 │
└────────────┘



In [29]:
con.sql("""
    CREATE OR REPLACE TABLE clean_customers AS
    SELECT *
    FROM raw_customers
""")

CountBIGINT
99441


In [30]:
con.sql("""
    SELECT COUNT(*) AS clean_rows
    FROM clean_customers
""").show()

┌────────────┐
│ clean_rows │
│   int64    │
├────────────┤
│      99441 │
└────────────┘



In [31]:
con.sql("""
    CREATE OR REPLACE TABLE clean_order_items AS
    SELECT *
    FROM raw_order_items
""")

CountBIGINT
112650


In [32]:
con.sql("""
    SELECT COUNT(*) AS clean_rows
    FROM clean_order_items
""").show()

┌────────────┐
│ clean_rows │
│   int64    │
├────────────┤
│     112650 │
└────────────┘



In [33]:
con.sql("""
    CREATE OR REPLACE TABLE clean_sellers AS
    SELECT *
    FROM raw_sellers
""")

CountBIGINT
3095


In [34]:
con.sql("""
    SELECT COUNT(*) AS clean_rows
    FROM clean_sellers
""").show()

┌────────────┐
│ clean_rows │
│   int64    │
├────────────┤
│       3095 │
└────────────┘



In [35]:
con.sql("""
    CREATE OR REPLACE TABLE clean_geolocation AS
    SELECT *
    FROM raw_geolocation
""")

CountBIGINT
1000163


In [36]:
con.sql("""
    SELECT COUNT(*) AS clean_rows
    FROM clean_geolocation
""").show()

┌────────────┐
│ clean_rows │
│   int64    │
├────────────┤
│    1000163 │
└────────────┘



In [37]:
con.sql("""
    CREATE OR REPLACE TABLE clean_product_category_name_translation AS
    SELECT *
    FROM raw_product_category_name_translation
""")

CountBIGINT
71


In [38]:
con.sql("""
    SELECT COUNT(*) AS clean_rows
    FROM clean_product_category_name_translation
""").show()

┌────────────┐
│ clean_rows │
│   int64    │
├────────────┤
│         71 │
└────────────┘



In [39]:
con.sql("""
    SELECT
        'orders' AS table_name,
        COUNT(*) AS rows
    FROM clean_orders

    UNION ALL

    SELECT
        'order_items',
        COUNT(*)
    FROM clean_order_items

    UNION ALL

    SELECT
        'order_payments',
        COUNT(*)
    FROM clean_order_payments

    UNION ALL

    SELECT
        'customers',
        COUNT(*)
    FROM clean_customers

    UNION ALL

    SELECT
        'products',
        COUNT(*)
    FROM clean_products

    UNION ALL

    SELECT
        'sellers',
        COUNT(*)
    FROM clean_sellers

    UNION ALL

    SELECT
        'order_reviews',
        COUNT(*)
    FROM clean_order_reviews

    UNION ALL

    SELECT
        'geolocation',
        COUNT(*)
    FROM clean_geolocation

    UNION ALL

    SELECT
        'category_translation',
        COUNT(*)
    FROM clean_product_category_name_translation
""").show()

┌──────────────────────┬─────────┐
│      table_name      │  rows   │
│       varchar        │  int64  │
├──────────────────────┼─────────┤
│ orders               │   99441 │
│ order_items          │  112650 │
│ order_payments       │  103883 │
│ customers            │   99441 │
│ products             │   32951 │
│ sellers              │    3095 │
│ order_reviews        │   99224 │
│ geolocation          │ 1000163 │
│ category_translation │      71 │
└──────────────────────┴─────────┘



In [40]:
con.sql("""
    SELECT COUNT(*) AS invalid_payments
    FROM clean_order_payments
    WHERE payment_value = 0
      AND payment_type = 'not_defined'
""").show()

┌──────────────────┐
│ invalid_payments │
│      int64       │
├──────────────────┤
│                0 │
└──────────────────┘



In [41]:
con.sql("""
    SELECT COUNT(*) AS unknown_categories
    FROM clean_products
    WHERE product_category_name = 'unknown'
""").show()

┌────────────────────┐
│ unknown_categories │
│       int64        │
├────────────────────┤
│                610 │
└────────────────────┘



In [42]:
con.close()